# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Preprocesadores.
# Para arboles: solo imputacion por mediana (no necesitan escalado).
# Todas las features son numericas, asi que aplicamos a todas.
numeric_features = feature_columns

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
    ]
)

# Para modelos sensibles a escala: imputacion + StandardScaler.
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
    ]
)
print("Preprocesadores creados.")


Preprocesadores creados.


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si escalamos **antes** de la validación cruzada, el `StandardScaler`
> aprendería la media y la desviación usando **también** las filas que luego
> harán de validación en cada fold. Eso es **fuga de información (data
> leakage)**: el modelo "ve" estadísticas del conjunto de validación durante el
> entrenamiento y las métricas de CV salen optimistas (no reflejan el
> desempeño con datos nuevos). Metiéndolo **dentro del `Pipeline`**, en cada
> fold el escalador se ajusta solo con el *train* de ese fold y se aplica al
> *validation* de ese fold, igual que ocurrirá en producción con datos nunca
> vistos.


## 3. Modelos candidatos

In [5]:
# Pipelines candidatos: cada uno con su preprocesador adecuado.
models = {
    "decision_tree": Pipeline(steps=[
        ("prep", clone(tree_preprocessor)),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline(steps=[
        ("prep", clone(scaled_preprocessor)),
        ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline(steps=[
        ("prep", clone(scaled_preprocessor)),
        ("clf", KNeighborsClassifier()),
    ]),
}
print("Modelos candidatos:", list(models))


Modelos candidatos: ['decision_tree', 'logistic_regression', 'knn']


## 4. Comparación inicial con validación cruzada

In [6]:
# Comparacion inicial con StratifiedKFold (5 folds) y cross_validate.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

initial_comparison = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring="accuracy",
        return_train_score=True,
    )
    initial_comparison.append({
        "model": name,
        "train_acc_mean": scores["train_score"].mean(),
        "val_acc_mean": scores["test_score"].mean(),
        "val_acc_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(initial_comparison).sort_values(
    "val_acc_mean", ascending=False
).reset_index(drop=True)
initial_comparison


,model,train_acc_mean,val_acc_mean,val_acc_std
0,logistic_regression,1.0000,0.979064,0.027713
1,knn,0.9771,0.950739,0.035274
2,decision_tree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [7]:
# Optimizacion de hiperparametros con GridSearchCV (espacios pequenos).
param_grids = {
    "decision_tree": {
        "clf__max_depth": [2, 3, 5, None],
        "clf__min_samples_leaf": [1, 2, 5],
    },
    "logistic_regression": {
        "clf__C": [0.1, 1.0, 10.0],
        "clf__penalty": ["l2"],
    },
    "knn": {
        "clf__n_neighbors": [3, 5, 7, 9],
        "clf__weights": ["uniform", "distance"],
    },
}

searches = {}
tuned_rows = []
for name, pipe in models.items():
    gs = GridSearchCV(
        pipe, param_grids[name],
        cv=cv, scoring="accuracy", n_jobs=-1,
    )
    gs.fit(X_train, y_train)
    searches[name] = gs
    tuned_rows.append({
        "model": name,
        "best_cv_acc": gs.best_score_,
        "best_params": gs.best_params_,
    })

tuned_comparison = pd.DataFrame(tuned_rows).sort_values(
    "best_cv_acc", ascending=False
).reset_index(drop=True)
tuned_comparison


/projects/sandbox/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/projects/sandbox/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/projects/sandbox/.venv/lib/python3.11/s

,model,best_cv_acc,best_params
0,logistic_regression,0.979064,"{'clf__C': 1.0, 'clf__penalty': 'l2'}"
1,knn,0.957882,"{'clf__n_neighbors': 7, 'clf__weights': 'unifo..."
2,decision_tree,0.902463,"{'clf__max_depth': 5, 'clf__min_samples_leaf': 2}"


## 6. Selección y persistencia del campeón

In [8]:
# Seleccion del campeon: mayor best_score_ de CV.
champion_name = max(searches, key=lambda n: searches[n].best_score_)
champion_search = searches[champion_name]
champion = champion_search.best_estimator_

print("Campeon:", champion_name)
print("CV accuracy:", round(champion_search.best_score_, 4))
print("Mejores hiperparametros:", champion_search.best_params_)

# Persistencia del pipeline completo + metadatos + resultados de CV.
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "cv_accuracy": float(champion_search.best_score_),
    "best_params": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
                     for k, v in champion_search.best_params_.items()},
    "random_state": RANDOM_STATE,
    "n_features": len(feature_columns),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2), encoding="utf-8"
)

tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("Guardados champion_model.joblib, training_metadata.json y cv_results.csv")


Campeon: logistic_regression
CV accuracy: 0.9791
Mejores hiperparametros: {'clf__C': 1.0, 'clf__penalty': 'l2'}
Guardados champion_model.joblib, training_metadata.json y cv_results.csv


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se eligió el modelo con mayor **accuracy de validación cruzada** entre las
> tres familias, usando siempre pipelines (sin fuga de información) y 5-fold
> estratificado. En el dataset Wine, los tres modelos alcanzan un desempeño muy
> alto porque las clases son bastante separables; el campeón combina **mejor
> accuracy de CV con baja variabilidad entre folds** (desviación estándar
> pequeña), lo que indica estabilidad y no un buen resultado por azar. Se tuvo
> en cuenta además el **sobreajuste** (comparando train vs. validation
> accuracy): el árbol tiende a sobreajustar si se deja sin podar, por lo que si
> un modelo escalado (regresión logística / KNN) iguala o supera su CV con
> menor brecha train-val, se prefiere por **mayor capacidad de
> generalización**. La regresión logística aporta además **interpretabilidad**
> (coeficientes) y simplicidad, criterios de desempate ante desempeños
> similares.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.
